In [0]:
import requests
import json

url = "https://stream.wikimedia.org/v2/stream/recentchange"

headers = {
    "User-Agent": "databricks-stream-demo"
}

response = requests.get(url, headers=headers, stream=True)

count = 0
limit = 20
records = []


for line in response.iter_lines():
    if line:
        decoded = line.decode("utf-8")

        if decoded.startswith("data:"):
            try:
                json_str = decoded[6:]   # remove "data: "
                data = json.loads(json_str)

                if data.get("wiki") == "enwiki":
                    records.append(data)
                    


                    count += 1
                    if count >= limit:
                        break
            except:
                pass




In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text(
    "catalog_name",
    "real_time_wikipedia_streaming_pipeline",
    "Catalog Name"
)
catalog_name = dbutils.widgets.get("catalog_name")


In [0]:
# %py
spark.sql(f"USE CATALOG `{catalog_name}`")


spark.sql("USE SCHEMA bronze")

spark.sql("CREATE VOLUME IF NOT EXISTS wikipedia_raw_data")




In [0]:
import datetime

current_date = datetime.datetime.now().strftime("%Y-%m-%d")
d = dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/wikipedia_raw_data/wikipedia_raw_data_{current_date}.json",
    json.dumps(records),
    overwrite=True,
)

display(spark.read.json(f"/Volumes/{catalog_name}/bronze/wikipedia_raw_data/wikipedia_raw_data_{current_date}.json"))

In [0]:
%sql
DROP TABLE IF EXISTS real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data;
DROP TABLE IF EXISTS real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data_vw;